# Benchmark the complete Lily dataset

This notebook benchmarks parity-game solvers and forced multistrategies. It can run all paper datasets and algorithms, including Chain-MS, and generates three aggregate tables from the existing `all_results` dataframe. An optional selected run is available without replacing the full results. Timing is run without `tracemalloc` so memory tracing does not distort runtime; BDD node counts are reported for symbolic solvers.

The generic runners also support one in-memory game (`benchmark_game`), any named benchmark set (`benchmark_dataset`), or all seven paper datasets (`benchmark_all_datasets`). Dataset runs use four worker processes by default and print a completion line as each solver run finishes; pass `workers=1` for sequential execution, `progress=False` to silence updates, or a callback to collect structured `BenchmarkProgress` events. Parallel runs use more peak memory, while each worker releases solver state between tasks. Custom methods can be passed as a `{name: solver_callable}` mapping.

In [1]:
from pathlib import Path
import sys

search_roots = (Path.cwd(), *Path.cwd().parents)
CODE = next(
    (candidate for root in search_roots
     for candidate in (root, root / 'code')
     if (candidate / 'src' / 'parity_bdd_lab').is_dir()),
    None,
)
if CODE is None:
    raise FileNotFoundError(f'Could not locate the code project from {Path.cwd()}')
sys.path.insert(0, str(CODE / 'src'))

import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
from parity_bdd_lab import benchmark_dataset, records_to_dataframe

WORKERS = 4

## Run every Lily game

Set `REPEATS` in the next cell to control how many records are collected per game/method pair; the notebook summarizes repeats by mean and standard deviation. Parallel progress is reported as runs finish; results are returned in dataset order. Memory tracing is optional and disabled here.

In [2]:
REPEATS = 3
METHODS = ('FPJ', 'Zielonka', 'CQ-MS-Cached', 'CQ-MS', 'Chain-MS')

run_records = benchmark_dataset(
    'Lily',
    methods=METHODS,
    repeats=REPEATS,
    measure_memory=False,
    workers=WORKERS,
)
run_results = records_to_dataframe(run_records)
assert run_results['game'].nunique() == 23
assert len(run_results) == 23 * len(METHODS) * REPEATS

# Check that the methods agree on every run, then aggregate runtimes per game/method.
agreement = run_results.groupby('game')['solution_fingerprint'].nunique()
assert (agreement == 1).all(), agreement[agreement != 1]
results = (
    run_results.groupby(['game', 'method'], as_index=False)
    .agg(
        mean_seconds=('runtime_seconds', 'mean'),
        std_seconds=('runtime_seconds', 'std'),
        even_winning_vertices=('even_winning_vertices', 'first'),
        odd_winning_vertices=('odd_winning_vertices', 'first'),
        strategy_edges=('strategy_edges', 'first'),
        bdd_nodes=('bdd_nodes', 'first'),
        solution_fingerprint=('solution_fingerprint', 'first'),
    )
)
display(results.sort_values(['game', 'method']).reset_index(drop=True))

[1/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / Zielonka run 1/3 complete
[2/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / Zielonka run 2/3 complete
[3/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / Zielonka run 3/3 complete
[4/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / CQ-MS-Cached run 1/3 complete
[5/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / FPJ run 1/3 complete
[6/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / FPJ run 2/3 complete
[7/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / FPJ run 3/3 complete
[8/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / CQ-MS-Cached run 2/3 complete
[9/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / CQ-MS-Cached run 3/3 complete
[10/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / CQ-MS run 1/3 complete
[11/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / CQ-MS run 2/3 complete
[12/345 runs; game 1/23] Lily / lilydemo01.tlsf.ehoa.oink / CQ-MS run 3/

,game,method,mean_seconds,std_seconds,even_winning_vertices,odd_winning_vertices,strategy_edges,bdd_nodes,solution_fingerprint
0,lilydemo01.tlsf.ehoa.oink,CQ-MS,0.010715,3.472647e-04,0,36,10,122.0,7f912123c2cfef2bb051d459bf0433ecdb6d2527a782e7...
1,lilydemo01.tlsf.ehoa.oink,CQ-MS-Cached,0.035025,4.363511e-02,0,36,10,122.0,7f912123c2cfef2bb051d459bf0433ecdb6d2527a782e7...
2,lilydemo01.tlsf.ehoa.oink,Chain-MS,0.009639,9.861578e-04,0,36,6,84.0,7f912123c2cfef2bb051d459bf0433ecdb6d2527a782e7...
3,lilydemo01.tlsf.ehoa.oink,FPJ,0.111180,2.716594e-04,0,36,7,170.0,7f912123c2cfef2bb051d459bf0433ecdb6d2527a782e7...
4,lilydemo01.tlsf.ehoa.oink,Zielonka,0.000085,4.013504e-05,0,36,6,NaN,7f912123c2cfef2bb051d459bf0433ecdb6d2527a782e7...
5,lilydemo02.tlsf.ehoa.oink,CQ-MS,0.008734,7.380344e-04,0,36,10,122.0,7f912123c2cfef2bb051d459bf0433ecdb6d2527a782e7...
6,lilydemo02.tlsf.ehoa.oink,CQ-MS-Cached,0.009678,1.096956e-03,0,36,10,122.0,7f912123c2cfef2bb051d459bf0433ecdb6d2527a782e7...
7,lilydemo02.tlsf.ehoa.oink,Chain-MS,0.009683,4.874124e-04,0,36,6,84.0,7f912123c2cfef2bb051d459bf0433ecdb6d2527a782e7...
8,lilydemo02.tlsf.ehoa.oink,FPJ,0.003783,4.566290e-04,0,36,7,170.0,7f912123c2cfef2bb051d459bf0433ecdb6d2527a782e7...
9,lilydemo02.tlsf.ehoa.oink,Zielonka,0.000065,1.334355e-05,0,36,6,NaN,7f912123c2cfef2bb051d459bf0433ecdb6d2527a782e7...


## Aggregate by method

These totals sum the five-run means over all Lily games and show the median and maximum per-game mean runtime.

In [3]:
method_summary = (
    results.groupby('method', as_index=False)
    .agg(
        games=('game', 'nunique'),
        total_mean_seconds=('mean_seconds', 'sum'),
        median_game_mean_seconds=('mean_seconds', 'median'),
        max_game_mean_seconds=('mean_seconds', 'max'),
        total_even_wins=('even_winning_vertices', 'sum'),
        total_odd_wins=('odd_winning_vertices', 'sum'),
        total_strategy_edges=('strategy_edges', 'sum'),
    )
)
display(method_summary)

,method,games,total_mean_seconds,median_game_mean_seconds,max_game_mean_seconds,total_even_wins,total_odd_wins,total_strategy_edges
0,CQ-MS,23,64.774108,0.012631,39.680519,7465,1936,10181
1,CQ-MS-Cached,23,1.132431,0.009903,0.553612,7465,1936,10181
2,Chain-MS,23,0.525840,0.009038,0.165603,7465,1936,7184
3,FPJ,23,0.819779,0.006569,0.271656,7465,1936,9127
4,Zielonka,23,0.030834,0.000247,0.015064,7465,1936,7108


## Other scopes

The all-dataset benchmark cell below runs only when `all_results` is not already present in the kernel. If your full results are already in memory, keep them and run the three table cells at the end. Set `RERUN_ALL_DATASETS = True` to explicitly refresh them.


In [ ]:
# One game already loaded as a ParityGame:
# from parity_bdd_lab import benchmark_game, read_pg
# game_records = benchmark_game(read_pg(CODE / 'data/games/image_example.pg',
#                                    vertex_labels=dict(enumerate('abcdefghi')),
#                                    name='image_example'),
#                                methods=('DFI', 'DFI-ns', 'FPJ', 'Zielonka'))

from parity_bdd_lab import benchmark_all_datasets
RERUN_ALL_DATASETS = False
if RERUN_ALL_DATASETS or 'all_results' not in globals():
    all_records = benchmark_all_datasets(
        methods=('FPJ', 'Zielonka', 'CQ-MS', 'CQ-MS-Cached', 'Chain-MS'),
        repeats=1,
        workers=WORKERS,
    )
    all_results = records_to_dataframe(all_records)
display(all_results)


[1/595 runs; game 1/119] Lily / lilydemo01.tlsf.ehoa.oink / Zielonka run 1/1 complete
[2/595 runs; game 1/119] Lily / lilydemo01.tlsf.ehoa.oink / Chain-MS run 1/1 complete
[3/595 runs; game 1/119] Lily / lilydemo01.tlsf.ehoa.oink / FPJ run 1/1 complete
[4/595 runs; game 1/119] Lily / lilydemo01.tlsf.ehoa.oink / CQ-MS-Cached run 1/1 complete
[5/595 runs; game 1/119] Lily / lilydemo01.tlsf.ehoa.oink / CQ-MS run 1/1 complete
[6/595 runs; game 2/119] Lily / lilydemo02.tlsf.ehoa.oink / Zielonka run 1/1 complete
[7/595 runs; game 2/119] Lily / lilydemo02.tlsf.ehoa.oink / FPJ run 1/1 complete
[8/595 runs; game 2/119] Lily / lilydemo02.tlsf.ehoa.oink / CQ-MS run 1/1 complete
[9/595 runs; game 2/119] Lily / lilydemo02.tlsf.ehoa.oink / CQ-MS-Cached run 1/1 complete
[10/595 runs; game 3/119] Lily / lilydemo03.tlsf.ehoa.oink / FPJ run 1/1 complete
[11/595 runs; game 2/119] Lily / lilydemo02.tlsf.ehoa.oink / Chain-MS run 1/1 complete
[12/595 runs; game 3/119] Lily / lilydemo03.tlsf.ehoa.oink / Ziel

In [ ]:
all_results

## Optional selected benchmark run

The three tables below use the existing `all_results` from the all-dataset run above. Leave `RUN_SELECTED_BENCHMARK = False` to preserve those full results. Set it to `True` only when you want a separate selected run; its output is stored in `selected_results` and does not replace `all_results`.


In [ ]:
from parity_bdd_lab import benchmark_selection, benchmark_dataset_names, records_to_dataframe

RUN_SELECTED_BENCHMARK = False
BENCHMARK_DATASETS = None  # e.g. 'Lily' or ['Lily', 'AMBA']; None selects all
BENCHMARK_METHODS = ['DFI', 'DFI-ns', 'FPJ', 'Zielonka', 'Chain-MS', 'CQ-MS', 'CQ-MS-Cached']
BENCHMARK_REPEATS = 1

if RUN_SELECTED_BENCHMARK:
    selected_records = benchmark_selection(
        BENCHMARK_DATASETS,
        methods=BENCHMARK_METHODS,
        repeats=BENCHMARK_REPEATS,
        workers=WORKERS,
    )
    selected_results = records_to_dataframe(selected_records)
    display(selected_results)

# Tables below always summarize the complete all_results dataframe.
_available_datasets = benchmark_dataset_names()
ACTIVE_DATASETS = [name for name in _available_datasets if name in set(all_results['dataset'])]
_preferred_methods = ['DFI', 'DFI-ns', 'FPJ', 'Zielonka', 'Chain-MS', 'CQ-MS', 'CQ-MS-Cached']
METHOD_ORDER = [name for name in _preferred_methods if name in set(all_results['method'])]


## Three aggregate tables for all benchmark results

These cells summarize the existing `all_results` dataframe from the all-dataset run above; they do not start another benchmark. Table 1 reports dataset graph statistics and the BDD node count for the static game encoding. Table 2 reports cumulative runtime by dataset and method. Table 3 reports mean returned BDD manager nodes by dataset and method (Zielonka has no BDD count).


In [ ]:
import gc
from parity_bdd_lab import benchmark_dataset_names, collect_bdd_garbage, iter_benchmark_files, read_pg
from parity_bdd_lab.fpj import _Encoding

# One row per game. Games and BDD managers are released after measuring each file.
game_statistics = []
for dataset in ACTIVE_DATASETS:
    for game_path in iter_benchmark_files(dataset):
        game = read_pg(game_path)
        vertices = game.vertices
        priorities = tuple(game.priority[v] for v in vertices)
        vertex_count = len(vertices)
        edge_count = sum(len(game.successors[v]) for v in vertices)

        encoding = _Encoding(game)
        collect_bdd_garbage(encoding.bdd)
        static_bdd_nodes = len(encoding.bdd)
        game_statistics.append({
            'dataset': dataset,
            'game': game_path.name,
            'vertices': vertex_count,
            'edges': edge_count,
            'min priority': min(priorities),
            'max priority': max(priorities),
            'static BDD nodes': static_bdd_nodes,
        })
        del encoding, game
        gc.collect()

game_statistics = pd.DataFrame(game_statistics)
dataset_summary_rows = []
for dataset in ACTIVE_DATASETS:
    games_in_set = game_statistics[game_statistics['dataset'] == dataset]
    dataset_summary_rows.append({
        'Set': dataset,
        '#games': games_in_set['game'].nunique(),
        'avg. n': games_in_set['vertices'].mean(),
        'max. n': games_in_set['vertices'].max(),
        'priorities': (
            f"{games_in_set['min priority'].min()}–{games_in_set['max priority'].max()}"
        ),
        'avg. out-degree': games_in_set['edges'].sum() / games_in_set['vertices'].sum(),
        'avg. #BDD nodes': games_in_set['static BDD nodes'].mean(),
    })

dataset_summary = pd.DataFrame(dataset_summary_rows)
display(dataset_summary.round({
    'avg. n': 0,
    'avg. out-degree': 2,
    'avg. #BDD nodes': 0,
}))


### Table 2: Cumulative runtime by dataset

Runtime is summed over all games within each repeat, then averaged across repeats. With a single run, the result is the cumulative time for that run.


In [ ]:
dataset_order = ACTIVE_DATASETS
method_order = METHOD_ORDER

per_repeat_runtime = (
    all_results
    .groupby(['dataset', 'method', 'run'], as_index=False)['runtime_seconds']
    .sum()
)
runtime_by_dataset = (
    per_repeat_runtime
    .groupby(['dataset', 'method'])['runtime_seconds']
    .mean()
    .unstack('method')
    .reindex(index=dataset_order, columns=method_order)
    .rename(columns={name: name.casefold() for name in method_order})
)
display(runtime_by_dataset.round(2))


### Table 3: Average returned BDD nodes by dataset

This uses the `bdd_nodes` values already stored in `all_results`: BDD nodes for DFI, DFI-ns, and FPJ, and chain-relation BDD nodes for Chain-MS. These are live manager sizes when each solver returns, not peak node counts. Zielonka does not use a decision diagram and appears blank.


In [ ]:
bdd_node_summary = (
    all_results
    .groupby(['dataset', 'method'])['bdd_nodes']
    .mean()
    .unstack('method')
    .reindex(index=dataset_order, columns=method_order)
    .rename(columns={name: name.casefold() for name in method_order})
)
display(bdd_node_summary.round(0))
